# VadCLIP + Nhất Quán Theo Dịch Chuyển — Vòng 2: Tối Ưu Cách Cắt Và Trọng Số

Vòng 1 (`docs/BaoCao_VadCLIP_Shift_Consistency.docx`) dựng xong cơ chế nhưng **chưa kết luận
được gì trên AUC**: chênh lệch giữa λ = 0,01 và λ = 0 là 0,63 điểm, trong khi hai lần chạy
λ = 0 — giống hệt nhau về mặt phương pháp — đã cách nhau 0,58 điểm. Bằng chứng duy nhất có giá
trị nằm ở phép đo độ nhạy dịch chuyển, nơi λ = 0,01 thắng ở cả ba mức dịch.

Vòng này sửa bốn thứ. Ba thứ đầu là lý do vòng 1 không đo được; thứ tư là lý do nó có thể chưa
từng thật sự bật phương pháp lên.

## 1. Luật chọn bộ trọng số

Vòng 1 chấm điểm khoảng 120 lần trên tập kiểm tra rồi giữ bộ trọng số có AUC cao nhất. Cách
này dùng tập kiểm tra hai lần — một lần để chọn, một lần để báo cáo — nên giá trị thu được nằm
cao hơn giá trị thật một cách hệ thống, và phần lớn chênh lệch giữa hai cấu hình chỉ phản ánh
lần chạy nào rút được lá bài may hơn.

Hướng khuếch đại theo lớp đã đo trực tiếp cái giá của luật này: bỏ nó đi làm **sai số đo giảm
khoảng một bậc độ lớn** (0,26 xuống 0,05 giữa hai lần chạy đối chứng). Vòng này chạy với
`--select-metric none`, tức lấy thẳng bộ trọng số cuối.

Kèm theo: `--eval-steps 0`, chỉ chấm điểm ở cuối mỗi epoch — 10 lần thay vì 120. Vừa nhanh hơn
nhiều, vừa không còn cám dỗ chọn đỉnh.

## 2. Liều augment hiện không đồng nhất giữa các video

Độ dịch Δ = 26 là **10% của lưới 256, không phải 10% của video**. Theo thống kê ở vòng 1,
**72% số tệp huấn luyện có độ dài dưới 256** (trung vị 138), nên chúng đi qua nhánh đệm không
chứ không bị nén. Với video trung vị, Δ = 26 là **19% độ dài thật**. Với một video dài bị nén,
26 ô lưới đại diện cho hàng nghìn hàng đặc trưng gốc.

Liều can thiệp vì thế biến thiên hơn hai bậc độ lớn giữa các video trong cùng một lô — đó là
phương sai bơm thẳng vào hàm mất mát, và là ứng viên hàng đầu giải thích vì sao hiệu ứng khó
đo. Vòng này thêm `--shift-ratio`: Δ = round(ρ · ℓ), với ℓ là độ dài hợp lệ của chính video đó.
Hệ quả miễn phí: **không còn video nào bị loại** vì thiếu vùng chồng lấn — vòng 1 mất 3,91% số
tệp, khoảng 626 video mỗi epoch.

## 3. Cách cắt chỉ ràng buộc đúng một khoảng dịch, và luôn cắt cùng một phía

| Cờ mới | Ý nghĩa |
|---|---|
| `--random-shift` | Lấy mẫu độ lớn trong [1, Δ] cho từng video thay vì luôn dùng giá trị lớn nhất. Một Δ cố định chỉ ràng buộc bất biến ở đúng một khoảng cách |
| `--shift-direction tail` | Đệm không vào **đầu** thay vì cắt đầu, tức đẩy nội dung về sau. Với 72% video ngắn hơn lưới, đây là phép **dời chỗ thuần tuý, không mất một hàng nội dung nào** — trong khi cắt đầu luôn phá huỷ Δ đoạn đầu |
| `--shift-direction both` | Rút phía cho từng video. Sự kiện bất thường của UCF-Crime thường nằm giữa video, nên cắt một phía làm lệch việc ngữ cảnh nào bị mất |
| `--shift-ratio-warmup` | Curriculum trên độ lớn phép dịch, song song với warm-up của λ. RandAugment cho thấy cường độ augment tối ưu **tăng đơn điệu** theo quá trình huấn luyện, nên một cường độ cố định sai ở một trong hai đầu |

## 4. Phát hiện quan trọng nhất: vòng 1 gần như chưa bật phương pháp lên

Đọc lại `Result/Result/logs_shift_consistency/train_v0.log` và tính phần đóng góp thật của số
hạng mới vào tổng hàm mục tiêu:

| Thời điểm | L1+L2+L3 | L_shift | λ·L_shift | **Tỉ lệ đóng góp** |
|---|---:|---:|---:|---:|
| epoch 1, bước 1280 | 3,2501 | 0,002127 | 2,13e-05 | **6,5e-06** |
| epoch 10, cuối | 0,7252 | 0,006311 | 6,31e-05 | **8,7e-05** |

Số hạng nhất quán chiếm khoảng **một phần trăm nghìn đến một phần mười nghìn** của hàm mục
tiêu. Đó không phải một lực tham gia vào quá trình tối ưu — đó là sai số làm tròn.

Điều này đổi cách đọc vòng 1 theo hai chiều. Chiều xấu: **λ = 0,01 chưa bao giờ là một phép thử
thật**, nên việc AUC không nhúc nhích không nói gì về phương pháp. Chiều tốt: một lực nhỏ đến
thế mà vẫn đẩy được chỉ số độ nhạy đi đúng hướng ở cả ba mức dịch, thì tín hiệu có thể mạnh hơn
nhiều khi λ nằm ở dải hợp lý.

Vòng này vì thế **không dò λ theo giá trị tuyệt đối** mà theo **tỉ lệ đóng góp**, qua
`--lambda-auto`: chạy 100 bước với số hạng tắt, đo cả hai vế, rồi giải ngược λ sao cho λ·L_shift
chiếm đúng tỉ lệ yêu cầu so với L1+L2+L3.

Cách này có một ưu điểm quyết định cho mục 8: khi **đổi cách cắt** thì độ lớn của L_shift đổi
theo, nên một λ tuyệt đối cố định sẽ làm phép so sánh giữa các cách cắt lẫn lộn với việc vô
tình đổi luôn trọng số hiệu dụng. Giữ tỉ lệ cố định thì không.

*Cảnh báo:* `--lambda-auto` chỉ đặt λ đúng bậc độ lớn. Tỉ lệ được giải ở bước 100 của epoch 1;
về sau L1 giảm khoảng 18 lần trong khi L_shift tăng khoảng 3 lần, nên tỉ lệ thực tế ở cuối quá
trình cao hơn mục tiêu khoảng một bậc. Giá trị λ đã giải được ghi vào cột `lambda_used` của CSV
để đối chiếu.

## Thiết kế quét, và ngưỡng quyết định đặt trước

**Chỉ số chính là độ nhạy dịch chuyển, không phải AUC.** Đó là chỗ vòng 1 đã cho tín hiệu nhất
quán, và là đại lượng mà hàm mất mát ràng buộc trực tiếp. AUC đóng vai trò **ràng buộc không
được xấu đi**, không phải mục tiêu.

| Mục | Số lần chạy | Mục đích |
|---|---:|---|
| §6 Đối chứng | 2 | λ = 0 ở hai seed. Cho **sàn nhiễu** dưới giao thức sạch — con số 0,58 của vòng 1 đo dưới luật chọn cũ và không còn dùng được |
| §7 Đường cong λ | 4 | Giữ nguyên cách cắt của vòng 1, quét tỉ lệ đóng góp 0,01 / 0,03 / 0,10 / 0,30. **Phần quan trọng nhất** — nó xác định dải λ có tác dụng |
| §8 Thang cắt | 4 | Ở λ tốt nhất của §7, cộng dồn từng cải tiến một: tỉ lệ, ngẫu nhiên, hai chiều, curriculum |
| §9 Xác nhận | 3 | `--consistency-detach`, và lặp lại cấu hình tốt nhất ở seed thứ hai |

Mười ba lần chạy 10 epoch. Nếu ngân sách hạn chế thì **§6 và §7 là tối thiểu** — chúng trả lời
câu hỏi lớn nhất còn treo lại từ vòng 1.

**Hình dạng cần tìm ở §7 là hình chuông, không phải đường đơn điệu.** Khác hẳn hệ số khuếch đại
μ của hướng kia, λ có trần thật: hàm mất mát nhất quán đạt cực tiểu tuyệt đối khi đầu ra hằng số
theo thời gian, mà nghiệm đó mâu thuẫn trực tiếp với cơ chế lấy k điểm cao nhất của MIL. Một
đường cong có đỉnh chứng minh dải khảo sát đã đủ; một đường tăng đều tới mức cuối nghĩa là chưa
chạm trần và phải quét thêm.

**Ngưỡng quyết định, đặt trước khi nhìn số:**

- Tương quan ở Δ = 16 tăng **hơn ba lần sàn nhiễu** đo ở §6, và AUC không tụt quá 0,5 điểm so
  với đối chứng, thì phương pháp có tác dụng thật.
- Tương quan tăng nhưng AUC tụt hơn 1 điểm ở mọi λ có tác dụng, thì đây là một **đánh đổi** và
  phải báo cáo là đánh đổi chứ không phải cải thiện.
- Không mức λ nào đẩy được tương quan quá sàn nhiễu, thì đó là kết luận âm tính, và nó **có giá
  trị**: bất biến theo dịch chuyển thời gian không phải ràng buộc mà bài toán này cần.

Kịch bản thứ ba có một đối chứng bên ngoài đáng đối chiếu: đã có công bố trên chính UCF-Crime
báo cáo rằng **xáo trộn ngẫu nhiên thứ tự thời gian của các vector đặc trưng không làm giảm độ
chính xác** (arXiv 2209.06435). Nếu điều đó đúng thì lượng thông tin nằm trên trục thời gian
trong benchmark này ít, và trần của mọi ràng buộc thuần tuý thời gian là thấp. Đó là lý do kịch
bản thứ ba không phải một thất bại.

## 1. Mount Drive Và Cấu Hình

Cell này **giả định bản code trên Drive đã được cập nhật** từ repo cục bộ. Các cờ
`--shift-ratio`, `--shift-direction`, `--shift-ratio-warmup`, `--lambda-auto`,
`--select-metric`, `--run-tag`, `--metrics-csv` đều là mới; bản cũ trên Drive không có chúng
và sẽ lỗi ngay ở lần chạy đầu. Mục 3 kiểm tra việc này **trước** khi tiêu tốn thời gian GPU.

Cần upload lại từ repo cục bộ: `VadCLIP/src/ucf_option_augment.py`,
`VadCLIP/src/ucf_train_augment.py`, `VadCLIP/src/utils/dataset_augment.py`, và thư mục
`VadCLIP/src/tests/`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT

# Chi dung lam moc doi chieu o muc 10, KHONG dung de khoi tao.
PAPER_MODEL = PROJECT_ROOT / 'model_ucf.pth'

RESULT_DIR = PROJECT_ROOT / 'Result'
LOG_DIR = RESULT_DIR / 'logs_shift_v2'
METRICS_CSV = str(RESULT_DIR / 'shift_v2_metrics.csv')

TRAIN_LIST = '../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST = '../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]

# Lich huan luyen giu nguyen nhu vong 1 de doi chung so sanh duoc.
# Chi GIAO THUC DO thay doi (select_metric, eval_steps).
MAX_EPOCH = 10
LR = '2e-5'
USE_PRETRAINED = False        # tu dau, giong vong 1
SELECT_METRIC = 'none'        # giu trong so CUOI
EVAL_STEPS = 0                # cham diem cuoi moi epoch: 10 lan thay vi ~120
LAMBDA_AUTO_STEPS = 100       # gan cuoi epoch 1 (125 buoc/epoch)

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
Path('model').mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, lambda_auto=0.0, lambda_consistency=0.0, seed=234,
                    shift_offset=26, shift_ratio=0.0, shift_direction='head',
                    random_shift=False, ratio_warmup=0, detach=False,
                    branch='c', warmup=1, extra=None):
    """Dung lenh huan luyen; tag quyet dinh toan bo ten file dau ra.

    lambda_consistency mac dinh 0: moi lan chay hoac dat lambda qua ty le dong gop
    (lambda_auto > 0), hoac la doi chung khong co so hang moi.
    """
    return PY + [
        'ucf_train_augment.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lambda-consistency', lambda_consistency,
        '--lambda-auto', lambda_auto,
        '--lambda-auto-steps', LAMBDA_AUTO_STEPS,
        '--shift-offset', shift_offset,
        '--shift-ratio', shift_ratio,
        '--shift-direction', shift_direction,
        '--random-shift', str(random_shift).lower(),
        '--shift-ratio-warmup', ratio_warmup,
        '--consistency-branch', branch,
        '--consistency-detach', str(detach).lower(),
        '--consistency-warmup', warmup,
        '--max-epoch', MAX_EPOCH,
        '--lr', LR,
        '--use-pretrained-model', str(USE_PRETRAINED).lower(),
        '--pretrained-model-path', PAPER_MODEL,
        '--num-workers', 4,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', f'model/model_{tag}.pth',
        '--checkpoint-path', f'model/checkpoint_{tag}.pth',
        '--save-cur-path', f'model/model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
    ] + list(extra or [])


def train_shift(tag, **kwargs):
    """Bo qua lan chay da co ket qua: runtime dut giua chung thi chay lai cell la tiep tuc."""
    if Path(f'model/model_{tag}.pth').exists():
        print(f'[bo qua] model/model_{tag}.pth da ton tai. Xoa file neu muon chay lai.')
        return None
    return run_command(build_train_cmd(tag, **kwargs), log_name=f'train_{tag}.log')


def shift_sensitivity(tag, model_path=None, offsets=(0, 8, 16, 32)):
    """Chi so CHINH cua vong nay: tuong quan diem so truoc va sau khi dich, nhanh C."""
    output_dir = RESULT_DIR / f'shift_sens_{tag}'
    if (output_dir / 'shift_sensitivity_summary.csv').exists():
        print(f'[bo qua] da co {output_dir}')
        return output_dir
    run_command(PY + [
        'ucf_shift_sensitivity.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--model-path', model_path or f'model/model_{tag}.pth',
        '--gt-path', '../list/gt_ucf.npy',
        '--offsets', *offsets,
        '--output-dir', output_dir,
    ], log_name=f'sens_{tag}.log')
    return output_dir


def analyze_per_class(tag, timeline_count=6):
    """Chi so theo lop. Dat va cham -- chi chay cho cau hinh thang cuoc."""
    output_dir = RESULT_DIR / f'perclass_{tag}'
    return run_command(PY + [
        'ucf_analyze_checkpoints.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--baseline-model-path', PAPER_MODEL,
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
        '--description-model-path', f'model/model_{tag}.pth',
        '--finetuned-model-type', 'baseline',
        '--output-dir', output_dir,
        *GT_ARGS,
        '--timeline-count', timeline_count,
    ], log_name=f'perclass_{tag}.log')


print('Project root :', PROJECT_ROOT)
print('Source dir   :', SRC_DIR)
print('Metrics CSV  :', METRICS_CSV)
print('Giao thuc do : select_metric =', SELECT_METRIC, '| eval_steps =', EVAL_STEPS)

## 2. Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas

## 3. Kiểm Tra File Và Kiểm Tra Phiên Bản Code

Cell này làm ba việc; hai việc sau quan trọng hơn việc đầu.

Thứ nhất, kiểm tra các file dữ liệu và script có mặt đủ chưa.

Thứ hai, **kiểm tra bản code trên Drive có phải bản mới không**, bằng cách hỏi thẳng
`ucf_option_augment.py` xem nó có biết các cờ mới hay không. Nếu thiếu, cell dừng ngay và in ra
danh sách cờ thiếu. Không có bước này thì lỗi chỉ lộ ra sau khi đã mount, cài đặt, giải nén
16.100 file đặc trưng và khởi động một lần chạy 10 epoch.

Thứ ba, **kiểm tra `utils/layers.py` có phải bản đã vá không**. Bản gốc hardcode `.to('cuda')`
trong `DistanceAdj.forward`, nên ma trận khoảng cách luôn nhảy sang GPU bất kể mô hình đang nằm
ở đâu. Trên một máy có GPU, hai file test dựng mô hình trên CPU sẽ chết với lỗi
*"Expected all tensors to be on the same device"*. Bản vá cho ma trận đó đi theo thiết bị của
chính module, và thêm cache vì nó là hằng số. Giá trị **không đổi một bit** — đã kiểm chứng
bằng so sánh trực tiếp với công thức gốc — nên kết quả huấn luyện trên GPU không bị ảnh hưởng.

Ba file `gt_*.npy` được kiểm tra riêng và không làm dừng cell, vì mục 4.1 sinh lại được — nhưng
việc đó cần feature nên phải làm sau mục 4.

In [ ]:
import importlib

required_paths = [
    SRC_DIR / 'model.py',
    SRC_DIR / 'ucf_train_augment.py',
    SRC_DIR / 'ucf_option_augment.py',
    SRC_DIR / 'ucf_shift_sensitivity.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'utils' / 'dataset_augment.py',
    SRC_DIR / 'utils' / 'layers.py',
    SRC_DIR / 'utils' / 'tools.py',
    SRC_DIR / 'tests' / 'test_dataset_augment.py',
    SRC_DIR / 'tests' / 'test_shift_consistency_loss.py',
    SRC_DIR / 'tests' / 'test_two_view_batching.py',
    SRC_DIR / 'tests' / 'test_train_smoke.py',
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    LIST_DIR / 'make_gt_ucf_relative.py',
]
gt_paths = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print('THIEU FILE:')
    for path in missing:
        print('  ', path)
    raise FileNotFoundError('Upload cac file con thieu len Drive roi chay lai cell nay.')

# Ban code tren Drive co phai ban moi khong?
import ucf_option_augment
importlib.reload(ucf_option_augment)
known = {action.dest for action in ucf_option_augment.parser._actions}
needed = {'shift_ratio', 'shift_direction', 'shift_ratio_warmup',
          'lambda_auto', 'lambda_auto_steps', 'select_metric', 'run_tag', 'metrics_csv'}
absent = sorted(needed - known)
if absent:
    print('BAN CODE TREN DRIVE LA BAN CU. Thieu cac tham so:')
    for name in absent:
        print('  --' + name.replace('_', '-'))
    raise RuntimeError(
        'Upload lai ucf_option_augment.py, ucf_train_augment.py, utils/dataset_augment.py '
        'va thu muc tests/ tu repo cuc bo.'
    )

# utils/layers.py co phai ban da va khong? Ban goc hardcode .to('cuda') trong
# DistanceAdj.forward, nen tren may co GPU moi test dung mo hinh tren CPU se chet voi
# "Expected all tensors to be on the same device". Kiem tra bang cach chay that.
import torch
from utils.layers import DistanceAdj

probe = DistanceAdj()                       # tham so nam tren CPU
probe_out = probe(2, 32)
if probe_out.device.type != 'cpu':
    raise RuntimeError(
        'utils/layers.py tren Drive la BAN CU: DistanceAdj tra ve tensor tren '
        f'{probe_out.device}, dang le phai theo thiet bi cua module (cpu). '
        'Upload lai VadCLIP/src/utils/layers.py tu repo cuc bo.'
    )
del probe, probe_out

GT_MISSING = [str(p) for p in gt_paths if not p.exists()]
print('Du toan bo file bat buoc, va ban code la ban moi.')
print('utils/layers.py: ban da va (DistanceAdj theo thiet bi cua module).')
print('Feature root ton tai:', DRIVE_FEATURE_ROOT.exists())
if GT_MISSING:
    print()
    print('Thieu ground truth (muc 4.1 se sinh lai):')
    for path in GT_MISSING:
        print('  ', path)

## 4. Copy Feature Sang Runtime Local — BẮT BUỘC

Feature trên Drive nằm ở dạng **file nén**, không phải thư mục. Cell này giải nén nó ra
`/content`. Không chạy cell này thì cả huấn luyện lẫn chấm điểm đều lỗi "missing feature files"
— đây là lỗi đã xảy ra một lần ở hướng nghiên cứu kia.

Chạy lại cell này sau **mỗi lần** runtime khởi động lại. Đọc trực tiếp từ Drive cũng chạy được
nhưng chậm hơn nhiều, và với mười ba lần chạy 10 epoch thì khác biệt là đáng kể.

In [ ]:
import shutil, time

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)
start = time.time()
if archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copying archive:', archive)
        shutil.copy2(archive, local_archive)
    print('Extracting:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    FEATURE_ROOT = LOCAL_FEATURE_ROOT
elif DRIVE_FEATURE_ROOT.exists():
    print('Khong tim thay file nen; dung truc tiep thu muc tren Drive (cham hon).')
    FEATURE_ROOT = DRIVE_FEATURE_ROOT
else:
    raise FileNotFoundError('Khong tim thay ca file nen lan thu muc feature tren Drive.')

print(f'Xong sau {time.time() - start:.0f}s. FEATURE_ROOT =', FEATURE_ROOT)
print('So file .npy:', sum(1 for _ in Path(FEATURE_ROOT).rglob("*.npy")))

### 4.1. Sinh Lại Ground Truth (chỉ khi mục 3 báo thiếu)

Ba file `gt_*.npy` không nằm trong repo — chúng được sinh từ `Temporal_Anomaly_Annotation.txt`
cộng với **độ dài thật của từng file đặc trưng**, nên bắt buộc phải có feature trước. Đó là lý
do cell này đứng sau mục 4.

Đừng dùng `list/make_gt_ucf.py` gốc: nó lọc theo `__0.npy` trong khi mọi file list test của dự
án dùng `__5.npy`, nên chạy ra file rỗng.

In [ ]:
if GT_MISSING:
    run_command(PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', str(LIST_DIR),
    ], log_name='make_gt.log')
else:
    print('Da co du ba file ground truth, bo qua.')

## 5. Unit Test

Bốn file, vài chục giây, chạy trên CPU với một bộ mã hoá CLIP giả nên không cần feature thật.

Hai bài quan trọng nhất với vòng này:

`test_fixed_mode_is_unchanged` khẳng định rằng khi để mặc định, cách cắt vẫn **y hệt vòng 1** —
kể cả chi tiết là video ngắn hơn Δ vẫn bị loại khỏi hàm mất mát. Nếu bài này hỏng thì đối chứng
của vòng này không so được với số liệu vòng 1.

`test_negative_offset_alignment_is_zero` khẳng định phép căn chỉnh cho độ dịch âm (hướng `tail`)
chính xác đến từng chỉ số. Sai một ô ở đây thì hàm mất mát sẽ phạt mô hình vì một sự bất đồng
không có thật, và mọi kết quả của mục 8 sẽ vô nghĩa mà không có dấu hiệu gì lộ ra.

Có bài nào FAIL thì dừng, đừng huấn luyện.

In [ ]:
for test_file in ('test_dataset_augment', 'test_shift_consistency_loss',
                  'test_two_view_batching', 'test_train_smoke'):
    run_command(PY + [f'tests/{test_file}.py'], log_name=f'{test_file}.log')
    print()

## 6. Đối Chứng — Và Sàn Nhiễu Của Vòng Này

Hai lần chạy với λ = 0. Về mặt toán học chúng là VadCLIP gốc: số hạng nhất quán vẫn được tính
và ghi log nhưng nhân với 0, nên nó không tham gia vào việc cập nhật trọng số.

**Chênh lệch giữa hai lần chạy này là sàn nhiễu**, và mọi con số ở các mục sau phải được đọc so
với nó. Con số 0,58 điểm AUC của vòng 1 đo dưới luật chọn checkpoint cũ nên **không dùng lại
được** — bỏ luật đó đi đã làm sai số của hướng nghiên cứu kia giảm khoảng một bậc độ lớn, và
chưa ai biết ở đây nó giảm bao nhiêu.

Hai lần chạy này khác nhau đúng một biến: seed. Đó là điều kiện để chênh lệch giữa chúng thật sự
là ước lượng của nhiễu chứ không phải của thứ gì khác.

**Đây là hai lần chạy không được bỏ.** Không có chúng thì không có thước đo, và mọi kết quả phía
sau chỉ là những con số không biết to hay nhỏ.

In [ ]:
train_shift('v2_ctrl_s234', lambda_auto=0.0, lambda_consistency=0.0, seed=234)
train_shift('v2_ctrl_s1234', lambda_auto=0.0, lambda_consistency=0.0, seed=1234)

## 7. Đường Cong λ — Phần Quan Trọng Nhất

Bốn lần chạy, **giữ nguyên cách cắt của vòng 1** (Δ = 26 cố định, cắt đầu, không ngẫu nhiên).
Biến duy nhất là tỉ lệ đóng góp mục tiêu của số hạng nhất quán: 0,01 rồi 0,03 rồi 0,10 rồi 0,30.

Vì sao dải này. Vòng 1 chạy ở tỉ lệ khoảng **6,5e-06 đến 8,7e-05** — xem mục 4 của phần mở đầu.
Mức thấp nhất ở đây, 0,01, đã lớn hơn vòng 1 khoảng **hai đến ba bậc độ lớn**. Nói cách khác,
toàn bộ dải này nằm hoàn toàn phía trên vùng vòng 1 đã thăm dò, và điểm λ = 0 của mục 6 khoá
chặt đầu dưới.

Vì sao dùng tỉ lệ chứ không phải giá trị λ tuyệt đối. Ở mục 8 cách cắt sẽ đổi, và khi đó độ lớn
của L_shift đổi theo — một cách cắt mạnh hơn sinh ra bất đồng lớn hơn. Nếu giữ λ tuyệt đối cố
định thì phép so sánh giữa các cách cắt bị lẫn với việc vô tình đổi luôn trọng số hiệu dụng.
Giữ **tỉ lệ đóng góp** cố định thì lực tác động lên quá trình tối ưu là như nhau, và cái được so
đúng là cách cắt.

Hình dạng cần tìm là **hình chuông**. Nếu tỉ lệ 0,30 vẫn tốt hơn 0,10 thì chưa chạm trần, chạy
thêm 1,0. Nếu 0,01 đã là tốt nhất thì đỉnh nằm thấp hơn, chạy thêm 0,003.

Mỗi lần chạy in một dòng `[lambda-auto]` cho biết giá trị λ đã giải được. Ghi lại nó — đó là cầu
nối giữa vòng này và giá trị 0,01 của vòng 1.

In [ ]:
LAMBDA_TARGETS = [0.01, 0.03, 0.10, 0.30]

for ratio in LAMBDA_TARGETS:
    print('=' * 100)
    train_shift(f'v2_lam{ratio:g}', lambda_auto=ratio, seed=234)

## 8. Thang Cắt — Cộng Dồn Từng Cải Tiến Một

Bốn lần chạy ở **tỉ lệ λ tốt nhất tìm được ở mục 7**. Đặt giá trị đó vào `BEST_RATIO` bên dưới
trước khi chạy cell.

Đây là một **thang cộng dồn**, không phải một lưới đầy đủ: mỗi bậc thêm đúng một thứ so với bậc
trước. Bốn lần chạy thay vì tám, và quan trọng hơn, nếu chỉ số đi lên đều qua bốn bậc thì bản
thân hình dạng đó là bằng chứng — nhiễu ngẫu nhiên không tạo ra một đường đơn điệu.

| Nhãn | Thêm gì so với bậc trước | Vì sao |
|---|---|---|
| `cut1_ratio` | Δ tính theo tỉ lệ độ dài video (ρ = 0,1) thay vì hằng số 26 | Đồng nhất hoá liều. Hiện Δ = 26 là 10% với video dài nhưng 19% với video trung vị, và không video nào bị loại nữa |
| `cut2_rand` | Lấy mẫu độ lớn trong [1, Δ] cho từng video | Δ cố định chỉ ràng buộc bất biến ở đúng một khoảng cách; lấy mẫu phủ toàn dải với cùng chi phí |
| `cut3_both` | Dịch cả hai chiều | Cắt đầu luôn phá huỷ phần mở đầu; đẩy về sau không mất nội dung nào với 72% video ngắn hơn lưới. Sự kiện của UCF-Crime thường nằm giữa video |
| `cut4_curr` | Curriculum: độ lớn tăng dần trong 3 epoch đầu | Cường độ augment tối ưu tăng theo quá trình huấn luyện, nên một giá trị cố định sai ở một trong hai đầu |

ρ = 0,1 được chọn để bậc đầu tiên gần nhất có thể với vòng 1: 10% của lưới so với 10% của video.
Khác biệt giữa `cut1_ratio` và lần chạy tương ứng ở mục 7 vì thế **cô lập đúng một thứ** — việc
liều có đồng nhất giữa các video hay không.

Lưu ý kỹ thuật: bật curriculum sẽ tự động tắt `persistent_workers`, vì tiến trình worker giữ một
bản sao đã pickle của dataset và sẽ không bao giờ thấy thay đổi độ lớn. Mỗi epoch vì thế tốn
thêm vài giây khởi động worker.

In [ ]:
# Dat gia tri tot nhat cua muc 7 vao day truoc khi chay.
BEST_RATIO = 0.10

common = dict(lambda_auto=BEST_RATIO, seed=234)

train_shift('v2_cut1_ratio', shift_ratio=0.1, **common)
train_shift('v2_cut2_rand',  shift_ratio=0.1, random_shift=True, **common)
train_shift('v2_cut3_both',  shift_ratio=0.1, random_shift=True,
            shift_direction='both', **common)
train_shift('v2_cut4_curr',  shift_ratio=0.1, random_shift=True,
            shift_direction='both', ratio_warmup=3, **common)

## 9. Xác Nhận — Neo Một Chiều Và Seed Thứ Hai

Ba lần chạy, đều dựa trên cấu hình thắng cuộc của mục 8. Đặt nó vào `BEST_CUT` bên dưới.

`detach` bật `--consistency-detach`: khung nhìn đầy đủ bị đóng băng thành một mục tiêu cố định,
nên ràng buộc trở thành một chiều thay vì hai chiều. Về mặt cơ học, điều này cắt bớt đường đi
tới nghiệm tầm thường — khung nhìn đầy đủ không còn bị kéo về phía khung nhìn dịch, nên áp lực
làm phẳng điểm số theo thời gian yếu hẳn đi. Đây là lựa chọn tiêu chuẩn trong dòng Mean Teacher
và có lý do rõ ràng để kỳ vọng nó cho phép λ lớn hơn mà không sập. Nếu nó thắng, đáng quét lại λ
quanh đó.

`best_s1234` lặp lại cấu hình thắng cuộc ở seed thứ hai. **Bắt buộc**: một cải thiện không lặp
lại được ở seed khác thì không phải cải thiện. Nó ghép cặp với `v2_ctrl_s1234` của mục 6.

`lam_hi` đẩy tỉ lệ lên gấp ba lần cấu hình thắng cuộc, để nhìn thấy phía dốc xuống của đường cong
ở đúng cách cắt tốt nhất. Đường cong λ ở mục 7 đo trên cách cắt cũ; cách cắt mới có thể dời đỉnh.

In [ ]:
# Dat cau hinh thang cuoc cua muc 8 vao day.
BEST_CUT = dict(shift_ratio=0.1, random_shift=True, shift_direction='both')

train_shift('v2_detach', lambda_auto=BEST_RATIO, seed=234, detach=True, **BEST_CUT)
train_shift('v2_best_s1234', lambda_auto=BEST_RATIO, seed=1234, **BEST_CUT)
train_shift('v2_lam_hi', lambda_auto=BEST_RATIO * 3, seed=234, **BEST_CUT)

## 10. Chấm Điểm Độ Nhạy Dịch Chuyển — Chỉ Số Chính

Phép đo này trả lời trực tiếp câu hỏi mà hàm mất mát nhắm tới: dịch đầu vào đi Δ bước, căn chỉnh
kết quả trở lại, thì chuỗi điểm số có giữ nguyên không?

Nó chạy trên 266 video kiểm tra (loại 24 video ngắn hơn độ dịch lớn nhất) và báo cáo tương quan
trung bình giữa chuỗi điểm gốc và chuỗi điểm sau khi dịch, trên nhánh C. Giá trị càng gần 1 càng
ổn định.

Đây là chỉ số chính vì hai lý do. Nó đo đúng tính chất mà hàm mất mát ràng buộc, nên nhạy hơn
AUC với can thiệp này rất nhiều. Và ở vòng 1 nó là chỗ duy nhất tín hiệu hiện ra nhất quán theo
cùng một hướng ở cả ba mức dịch, trong khi AUC và mAP nằm gọn trong biên độ nhiễu.

Mốc `paper` là checkpoint của tác giả, để đối chiếu với Bảng 5 của báo cáo vòng 1 (0,734 / 0,697
/ 0,638 và biên độ dao động 0,152). Nó không phải mục tiêu — chỉ là một điểm neo cho biết thang
đo chưa trôi.

Cell tự bỏ qua những lần chạy chưa tồn tại, nên chạy được ngay cả khi mới xong mục 6 và 7.

In [ ]:
ALL_TAGS = [
    'v2_ctrl_s234', 'v2_ctrl_s1234',
    'v2_lam0.01', 'v2_lam0.03', 'v2_lam0.1', 'v2_lam0.3',
    'v2_cut1_ratio', 'v2_cut2_rand', 'v2_cut3_both', 'v2_cut4_curr',
    'v2_detach', 'v2_best_s1234', 'v2_lam_hi',
]

shift_sensitivity('paper', model_path=PAPER_MODEL)
for tag in ALL_TAGS:
    if Path(f'model/model_{tag}.pth').exists():
        print('=' * 100)
        shift_sensitivity(tag)
    else:
        print(f'[chua co] {tag}')

### 10.1. Bảng Tổng Hợp

Cell này ghép hai nguồn: dòng cuối cùng của mỗi lần chạy trong CSV huấn luyện (AUC, AP, giá trị
λ đã giải), và bảng độ nhạy dịch chuyển tương ứng.

Nó in ba thứ.

**Bảng chính** — mỗi lần chạy một dòng, với tương quan ở ba mức dịch và AUC. Cột `d_corr16` là
chênh lệch tương quan ở Δ = 16 so với đối chứng cùng seed; đó là con số cần nhìn trước tiên.

**Sàn nhiễu** — chênh lệch giữa hai lần chạy đối chứng, trên đúng những đại lượng dùng để kết
luận. Mọi con số ở bảng chính phải được đọc so với nó.

**Kiểm tra ngưỡng quyết định** — áp thẳng ba tiêu chí đã đặt ra ở phần mở đầu, để câu trả lời
không phụ thuộc vào việc nhìn bảng số rồi tự thuyết phục mình.

In [ ]:
import pandas as pd

OFFSETS = [8, 16, 32]


def ctrl_of(tag):
    return 'v2_ctrl_s1234' if tag.endswith('_s1234') else 'v2_ctrl_s234'


def read_sensitivity(tag):
    path = RESULT_DIR / f'shift_sens_{tag}' / 'shift_sensitivity_summary.csv'
    if not path.exists():
        return None
    frame = pd.read_csv(path).set_index('offset')
    row = {f'corr{o}': float(frame.loc[o, 'mean_classifier_corr'])
           for o in OFFSETS if o in frame.index}
    row['auc_spread'] = float(frame.iloc[0]['classifier_auc_spread_common'])
    return row


# AUC / AP / lambda tu CSV huan luyen, lay dong cuoi cua moi lan chay.
train_metrics = {}
if Path(METRICS_CSV).exists():
    metrics = pd.read_csv(METRICS_CSV)
    final = metrics[metrics.is_final == 1] if 'is_final' in metrics else metrics
    for tag, group in final.groupby('run'):
        last = group.sort_values('step').iloc[-1]
        train_metrics[tag] = {'auc': float(last['auc']), 'ap': float(last['ap']),
                              'lambda_used': float(last['lambda_used'])}

rows = []
for tag in ['paper'] + ALL_TAGS:
    sens = read_sensitivity(tag)
    if sens is None:
        continue
    rows.append({'run': tag, **sens, **train_metrics.get(tag, {})})
table = pd.DataFrame(rows).set_index('run')

# Delta so voi doi chung CUNG SEED.
for tag in table.index:
    base = ctrl_of(tag)
    if tag == 'paper' or tag.startswith('v2_ctrl') or base not in table.index:
        continue
    for col in ('corr8', 'corr16', 'corr32', 'auc'):
        if col in table.columns:
            table.loc[tag, 'd_' + col] = table.loc[tag, col] - table.loc[base, col]

print('=== Bang chinh ===')
print(table.round(4).to_string())

print()
print('=== San nhieu (chenh lech giua hai lan chay doi chung) ===')
noise = {}
if {'v2_ctrl_s234', 'v2_ctrl_s1234'} <= set(table.index):
    for col in ('corr8', 'corr16', 'corr32', 'auc', 'auc_spread'):
        if col in table.columns:
            noise[col] = abs(table.loc['v2_ctrl_s234', col] - table.loc['v2_ctrl_s1234', col])
            print(f'  {col:<12} {noise[col]:.4f}')
else:
    print('  Chua chay du hai doi chung -- khong co thuoc do. Chay muc 6 truoc.')

print()
print('=== Nguong quyet dinh ===')
if 'corr16' in noise and 'd_corr16' in table.columns:
    floor = noise['corr16']
    candidates = table[table['d_corr16'].notna()]
    if len(candidates):
        best = candidates['d_corr16'].idxmax()
        gain = float(candidates.loc[best, 'd_corr16'])
        auc_cost = float(candidates.loc[best, 'd_auc'])
        print(f'  Cau hinh manh nhat : {best}')
        print(f'  d_corr16           : {gain:+.4f}  (san nhieu {floor:.4f}, '
              f'tuc {gain / floor if floor else float("inf"):.1f} lan)')
        print(f'  d_auc              : {auc_cost:+.2f} diem')
        if gain > 3 * floor and auc_cost > -0.5:
            print('  => TIEU CHI 1: phuong phap co tac dung that.')
        elif gain > 3 * floor and auc_cost <= -1.0:
            print('  => TIEU CHI 2: co tac dung nhung DANH DOI voi AUC. Bao cao la danh doi.')
        elif gain <= 3 * floor:
            print('  => TIEU CHI 3: khong vuot duoc san nhieu. Ket luan am tinh (van co gia tri).')
        else:
            print('  => Nam giua cac tieu chi: doc bang chinh va quyet dinh bang tay.')
else:
    print('  Chua du du lieu.')

### 10.2. Đường Cong λ Và Thang Cắt, Tách Riêng

Hai hình dạng cần nhìn tách khỏi bảng tổng hợp, vì bằng chứng nằm ở **hình dạng** chứ không ở
từng giá trị đơn lẻ.

Với đường cong λ, cái cần tìm là một đỉnh. Ba điểm cùng tăng thì chưa biết trần ở đâu; một đỉnh
rõ ràng thì dải khảo sát đã đủ và có thể kết luận.

Với thang cắt, cái cần tìm là tính đơn điệu qua bốn bậc. Từng bậc riêng lẻ gần như chắc chắn nằm
trong nhiễu — bốn bậc cùng đi lên thì không.

In [ ]:
NAN = float('nan')

print('=== Duong cong lambda (cach cat cua vong 1) ===')
print(f"{'ty le':>8} {'lambda giai duoc':>18} {'corr16':>9} {'d_corr16':>10} "
      f"{'AUC':>8} {'d_auc':>8}")
for ratio in [0.01, 0.03, 0.10, 0.30]:
    tag = f'v2_lam{ratio:g}'
    if tag not in table.index:
        continue
    r = table.loc[tag]
    print(f"{ratio:>8g} {r.get('lambda_used', NAN):>18.4e} {r.get('corr16', NAN):>9.4f} "
          f"{r.get('d_corr16', NAN):>+10.4f} {r.get('auc', NAN):>8.2f} "
          f"{r.get('d_auc', NAN):>+8.2f}")
print('  Moc: vong 1 chay o ty le ~6,5e-06 den 8,7e-05, tuc THAP HON ca diem thap nhat o day.')

print()
print('=== Thang cat (cong don) ===')
ladder = [(f'v2_lam{BEST_RATIO:g}', 'goc: offset 26 co dinh'),
          ('v2_cut1_ratio', '+ ty le theo do dai'),
          ('v2_cut2_rand', '+ do lon ngau nhien'),
          ('v2_cut3_both', '+ dich hai chieu'),
          ('v2_cut4_curr', '+ curriculum')]
print(f"{'cau hinh':>16} {'them gi':<26} {'corr16':>9} {'d_corr16':>10} {'AUC':>8} {'d_auc':>8}")
for tag, what in ladder:
    if tag not in table.index:
        continue
    r = table.loc[tag]
    print(f"{tag:>16} {what:<26} {r.get('corr16', NAN):>9.4f} {r.get('d_corr16', NAN):>+10.4f} "
          f"{r.get('auc', NAN):>8.2f} {r.get('d_auc', NAN):>+8.2f}")

print()
print('=== Lap lai o seed thu hai ===')
for a, b in [('v2_cut3_both', 'v2_best_s1234')]:
    if a in table.index and b in table.index:
        print(f"  {a:<16} d_corr16 {table.loc[a, 'd_corr16']:+.4f}")
        print(f"  {b:<16} d_corr16 {table.loc[b, 'd_corr16']:+.4f}")
        gap = abs(table.loc[a, 'd_corr16'] - table.loc[b, 'd_corr16'])
        print(f"  chenh lech giua hai seed: {gap:.4f}  (san nhieu {noise.get('corr16', NAN):.4f})")

## 11. Chỉ Số Theo Lớp Cho Cấu Hình Thắng Cuộc

Chỉ chạy cho **một** cấu hình — cái thắng ở mục 10. Phép đo này đắt và không dùng để chọn cấu
hình, nó dùng để hiểu cấu hình đã chọn.

Câu hỏi cụ thể đáng hỏi ở đây: những lớp mà sự kiện bất thường **ngắn và đột ngột** có phải là
những lớp hưởng lợi nhiều nhất không? Nếu ràng buộc theo thời gian có tác dụng thì đó là chỗ nó
phải hiện ra rõ nhất, vì một sự kiện chỉ chiếm vài ô lưới là thứ dễ bị lệch nhất khi dịch.
Explosion là ứng viên hàng đầu.

Bắt buộc kèm số video kiểm tra khi trình bày. Abuse có 2 video và Assault có 3; AUC theo lớp tính
trên số lượng như vậy dao động tới 3,9 điểm giữa các lần chạy đáng lẽ cho cùng kết quả, nên hai
lớp này phải bị loại khỏi mọi trung bình.

In [ ]:
WINNER = 'v2_cut3_both'   # doi thanh cau hinh thang cuoc that su

if Path(f'model/model_{WINNER}.pth').exists():
    analyze_per_class(WINNER)
else:
    print(f'Chua co model/model_{WINNER}.pth')

In [ ]:
TINY_CLASSES = {'Abuse', 'Assault'}   # 2 va 3 video test -> khong dung duoc

per_class_path = RESULT_DIR / f'perclass_{WINNER}' / 'per_class_metrics.csv'
if per_class_path.exists():
    pc = pd.read_csv(per_class_path)
    pc = pc[(pc.branch == 'classifier') & (pc.label != 'Normal')]
    pivot = pc.pivot_table(index='label', columns='run', values='auc')
    print(pivot.round(2).to_string())
    usable = [c for c in pivot.index if c not in TINY_CLASSES]
    print()
    print('Trung binh (bo Abuse va Assault):')
    print(pivot.loc[usable].mean().round(2).to_string())
    print()
    print('Luu y: Abuse (2 video) va Assault (3 video) khong dung duoc, da bi loai.')
else:
    print('Chua co', per_class_path)

## 12. Cách Đọc Kết Quả

**Chỉ số chính là `d_corr16`, không phải `d_auc`.** Ở vòng 1, AUC của mọi cấu hình nằm gọn trong
biên độ nhiễu và không phân biệt được cấu hình nào với cấu hình nào. AUC ở đây là ràng buộc
không được xấu đi, không phải mục tiêu. Nếu kết luận cuối cùng dựa vào một chênh lệch AUC dưới
sàn nhiễu thì nó không phải kết luận.

**Bằng chứng mạnh nhất là hình dạng, không phải giá trị.** Một đường cong λ có đỉnh, hoặc một
thang cắt đi lên đều qua bốn bậc, có sức nặng hơn bất kỳ chênh lệch đơn lẻ nào — vì nhiễu ngẫu
nhiên không tạo ra những hình dạng đó. Đây là bài học đắt nhất của hướng nghiên cứu kia, nơi mười
sáu lần chạy vòng 1 không kết luận được gì, còn đường đáp ứng đơn điệu qua bốn mức ở vòng 2 thì
kết luận được ngay.

**Kiểm tra giá trị λ đã giải được.** Nếu `lambda_used` ở tỉ lệ 0,10 rơi vào khoảng vài chục thì
điều đó xác nhận phân tích ở mục 4 phần mở đầu: vòng 1 chạy ở λ = 0,01, tức thấp hơn ba bậc độ
lớn, và chưa bao giờ thật sự thử phương pháp. Con số này đáng đưa vào báo cáo — nó giải thích
toàn bộ vòng 1 bằng một dòng.

**Cảnh giác với dấu hiệu sập về nghiệm tầm thường.** Nếu ở tỉ lệ cao mà `corr16` tăng vọt lên gần
1 trong khi AUC tụt mạnh, đó không phải thành công: mô hình đã học cách cho ra điểm số gần như
hằng số theo thời gian, thoả mãn hàm nhất quán một cách hoàn hảo và phá hỏng khả năng định vị.
Cột `auc_spread` và AUC tuyệt đối là thứ phát hiện ra điều này. Đó chính là trần mà mục 7 đi tìm.

**Nếu kết quả là âm tính, đừng chôn nó.** Có công bố trên chính UCF-Crime báo cáo rằng xáo trộn
thứ tự thời gian của các vector đặc trưng không làm giảm độ chính xác. Một kết quả âm tính ở đây,
đo cẩn thận với giao thức sạch và một dải λ đủ rộng, là bằng chứng độc lập cho cùng một điều: bài
toán này không khai thác trục thời gian nhiều như trực giác gợi ý. Kèm theo con số của mục 4 —
rằng vòng 1 chạy ở tỉ lệ 1e-5 — thì câu chuyện đầy đủ và tự nhất quán.

## Việc còn lại sau vòng này

Nếu mục 7 cho một đỉnh rõ và mục 8 cho một thang đơn điệu, ba việc tiếp theo theo thứ tự ưu tiên:

Seed thứ ba, để biến sàn nhiễu từ một khoảng giữa hai điểm thành ước lượng có biên độ.

Quét λ lại quanh đỉnh trên **cách cắt mới**, vì cách cắt mới có thể dời đỉnh đi.

Ràng buộc nhánh A (`--consistency-branch both`), để xem tính nhất quán có chuyển được sang mAP
hay không — mAP là chỉ số duy nhất mà vòng 1 thấy cải thiện rõ (15,79 lên 16,60 ở mAP@0,1).